In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.cuda.amp import GradScaler

# Import utils
from utils.Logger import Logger
from utils.Seed import set_seed
from utils.Splitter import stratified_split
from classes.FeatureDataset.WaveformFeatureDataset import WaveformFeatureDataset
from classes.FeatureDataset.CombinedFeatureDataset import CombinedFeatureDataset
from classes.FeatureDataset.ListDataset import ListDataset

# Import XLSR-Conformer with TCM components
from classes.models.XLSR_Conformer_TCM.model_XLSR_Conformer_TCM import XLSRConformerTCM
from classes.models.XLSR_Conformer_TCM.model_XLSR_Conformer_TCM_diff_pipeline import XLSRConformerTCMDiffPipeline
from classes.models.XLSR_Conformer_TCM.trainer_XLSR_Conformer_TCM import (
    test_xlsr_conformer_tcm,
    load_model_xlsr_conformer_tcm
)

seed = 42
set_seed(42)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ""

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
learning_rate = 0.000001
epochs = 30

main_keys = [
    "unseen all samples (UnID)"
]

# XLSR-Conformer with TCM

---
# Waveform (Baseline)
---

## Load Test Data

In [ ]:
# samples will be stored here:
xlsr_test_samples = {
    "unseen all samples (UnID)": [],
}

# unseen spoof datasets
xlsr_spoof_dupdub_notindataset_tts_dir = "test_preprocessed_data/waveform/Spoof/TTS/DupDub-NotInDataset"

# Bonafide datasets
xlsr_bonafide_commonvoice_dir = "preprocessed_data/waveform/Bonafide/CommonVoice"
xlsr_bonafide_prosa_dir = "preprocessed_data/waveform/Bonafide/Prosa"

## ------------------------------------
## UNSEEN SPOOF DATASETS
## ------------------------------------
# DupDub NotInDataset TTS
if os.path.exists(xlsr_spoof_dupdub_notindataset_tts_dir):
    xlsr_spoof_dupdub_notindataset_tts_dataset = WaveformFeatureDataset(xlsr_spoof_dupdub_notindataset_tts_dir, force_label=0)
    xlsr_spoof_dupdub_notindataset_tts_list = ListDataset([(features, 0) for features, _ in xlsr_spoof_dupdub_notindataset_tts_dataset.samples])
    xlsr_test_samples["unseen all samples (UnID)"].extend(xlsr_spoof_dupdub_notindataset_tts_list)

    print(f"Loaded {len(xlsr_spoof_dupdub_notindataset_tts_list)} samples from {xlsr_spoof_dupdub_notindataset_tts_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_spoof_dupdub_notindataset_tts_dir}")

### Bonafides
# Bonafide CommonVoice
if os.path.exists(xlsr_bonafide_commonvoice_dir):
    dataset_commonvoice = WaveformFeatureDataset(xlsr_bonafide_commonvoice_dir, force_label=1)
    xlsr_bonafide_commonvoice = ListDataset([(features, 1) for features, _ in dataset_commonvoice.samples])
    t_c, v_c, te_c = stratified_split(xlsr_bonafide_commonvoice, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_bonafide_commonvoice_list = ListDataset([xlsr_bonafide_commonvoice[i] for i in range(len(te_c))])
    xlsr_test_samples["unseen all samples (UnID)"].extend(xlsr_bonafide_commonvoice_list)

    print(f"Loaded {len(xlsr_bonafide_commonvoice_list)} samples from {xlsr_bonafide_commonvoice_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_bonafide_commonvoice_dir}")

# Bonafide Prosa
if os.path.exists(xlsr_bonafide_prosa_dir):
    dataset_prosa = WaveformFeatureDataset(xlsr_bonafide_prosa_dir, force_label=1)
    xlsr_bonafide_prosa = ListDataset([(features, 1) for features, _ in dataset_prosa.samples])
    t_p, v_p, te_p = stratified_split(xlsr_bonafide_prosa, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_bonafide_prosa_list = ListDataset([xlsr_bonafide_prosa[i] for i in range(len(te_p))])
    xlsr_test_samples["unseen all samples (UnID)"].extend(xlsr_bonafide_prosa_list)

    print(f"Loaded {len(xlsr_bonafide_prosa_list)} samples from {xlsr_bonafide_prosa_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_bonafide_prosa_dir}")

print("-----------------------------------------------")
print(f"Total unseen UnID samples: {len(xlsr_test_samples['unseen all samples (UnID)'])}")
print("-----------------------------------------------")

xlsr_test_samples_dataloaders = {
    key: DataLoader(value, batch_size=batch_size, shuffle=False, num_workers=4)
    for key, value in xlsr_test_samples.items()
}

## Test XLSR-Conformer TCM Model (Baseline)

In [ ]:
# Model configuration (same as training)
xlsr_model_config = {
    'emb_size': 144,           # Embedding size
    'num_encoders': 4,         # Number of conformer encoder blocks
    'heads': 4,                # Number of attention heads
    'kernel_size': 31,         # Kernel size for conv module
    'cp_path': 'xlsr2_300m.pt', # Path to pretrained XLSR model
    'fine_tune_ssl': True      # Whether to fine-tune SSL model
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Set up model, optimizer, and scaler ---
model = XLSRConformerTCM(xlsr_model_config, device).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
scaler = GradScaler()

# --- Load the trained model ---
checkpoint_path = r"pretrained_weights/waveform/XLSR_Conformer_TCM/xlsr_conformer_tcm_waveform-ep_30-bs_32-lr_0.000001.pth"

print(f"\nLoading model from: {checkpoint_path}")
start_epoch = load_model_xlsr_conformer_tcm(
    model, optimizer, scaler,
    path=checkpoint_path,
    device=device
)

print(f"Loaded checkpoint from epoch {start_epoch}")

# Count parameters
nb_params = sum([param.view(-1).size()[0] for param in model.parameters() if param.requires_grad])
print(f'Number of trainable parameters: {nb_params:,}')

In [ ]:
# Test the model
for key, test_sample in xlsr_test_samples_dataloaders.items():
    print(f"\n{'='*60}")
    print(f"Testing {key}")
    print(f"{'='*60}")
    
    if key in main_keys:
        predictions, targets, metrics = test_xlsr_conformer_tcm(model, test_sample, device=device)
        
        # Print summary
        print(f"\n--- Results Summary ---")
        print(f"Accuracy: {metrics['accuracy']:.2f}%")
        print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"F2 Score: {metrics['f2']:.4f}")
        print(f"EER: {metrics['eer']:.4f}")
        print(f"actDCF: {metrics['actDCF']:.4f}")
        print(f"minDCF: {metrics['minDCF']:.4f}")
        print(f"CLLR: {metrics['cllr']:.4f}")
    
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Testing completed!")
print(f"{'='*60}")

In [ ]:
# Clean up baseline model from GPU memory
print("Cleaning up baseline model from GPU memory...")

# Delete model components explicitly
if hasattr(model, "ssl_model"):
    del model.ssl_model
if hasattr(model, "ssl_encoder"):
    del model.ssl_encoder

del model
del optimizer
del scaler

# Force garbage collection
import gc
gc.collect()

# Clear CUDA cache multiple times
torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Print memory info
if torch.cuda.is_available():
    memory_allocated = torch.cuda.memory_allocated() / 1024**3
    memory_reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory allocated: {memory_allocated:.2f} GB")
    print(f"GPU memory reserved: {memory_reserved:.2f} GB")

print("GPU memory cleaned successfully!")

---
# Combined (Diff Pipeline)
---

## Load Test Data (Combined Features)

In [ ]:
# samples will be stored here:
xlsr_combined_test_samples = {
    "unseen all samples (UnID)": [],
}

# unseen spoof datasets
xlsr_combined_spoof_dupdub_notindataset_tts_dir = "test_preprocessed_data/combined/Spoof/TTS/DupDub-NotInDataset"

# Bonafide datasets
xlsr_combined_bonafide_commonvoice_dir = "preprocessed_data/combined/Bonafide/CommonVoice"
xlsr_combined_bonafide_prosa_dir = "preprocessed_data/combined/Bonafide/Prosa"

## ------------------------------------
## UNSEEN SPOOF DATASETS
## ------------------------------------
# DupDub NotInDataset TTS
if os.path.exists(xlsr_combined_spoof_dupdub_notindataset_tts_dir):
    xlsr_combined_spoof_dupdub_notindataset_tts_dataset = CombinedFeatureDataset(xlsr_combined_spoof_dupdub_notindataset_tts_dir, force_label=0)
    xlsr_combined_spoof_dupdub_notindataset_tts_list = ListDataset([(features, 0) for features, _ in xlsr_combined_spoof_dupdub_notindataset_tts_dataset.samples])
    xlsr_combined_test_samples["unseen all samples (UnID)"].extend(xlsr_combined_spoof_dupdub_notindataset_tts_list)

    print(f"Loaded {len(xlsr_combined_spoof_dupdub_notindataset_tts_list)} samples from {xlsr_combined_spoof_dupdub_notindataset_tts_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_combined_spoof_dupdub_notindataset_tts_dir}")

### Bonafides
# Bonafide CommonVoice
if os.path.exists(xlsr_combined_bonafide_commonvoice_dir):
    dataset_commonvoice = CombinedFeatureDataset(xlsr_combined_bonafide_commonvoice_dir, force_label=1)
    xlsr_combined_bonafide_commonvoice = ListDataset([(features, 1) for features, _ in dataset_commonvoice.samples])
    t_c, v_c, te_c = stratified_split(xlsr_combined_bonafide_commonvoice, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_combined_bonafide_commonvoice_list = ListDataset([xlsr_combined_bonafide_commonvoice[i] for i in range(len(te_c))])
    xlsr_combined_test_samples["unseen all samples (UnID)"].extend(xlsr_combined_bonafide_commonvoice_list)

    print(f"Loaded {len(xlsr_combined_bonafide_commonvoice_list)} samples from {xlsr_combined_bonafide_commonvoice_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_combined_bonafide_commonvoice_dir}")

# Bonafide Prosa
if os.path.exists(xlsr_combined_bonafide_prosa_dir):
    dataset_prosa = CombinedFeatureDataset(xlsr_combined_bonafide_prosa_dir, force_label=1)
    xlsr_combined_bonafide_prosa = ListDataset([(features, 1) for features, _ in dataset_prosa.samples])
    t_p, v_p, te_p = stratified_split(xlsr_combined_bonafide_prosa, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_combined_bonafide_prosa_list = ListDataset([xlsr_combined_bonafide_prosa[i] for i in range(len(te_p))])
    xlsr_combined_test_samples["unseen all samples (UnID)"].extend(xlsr_combined_bonafide_prosa_list)

    print(f"Loaded {len(xlsr_combined_bonafide_prosa_list)} samples from {xlsr_combined_bonafide_prosa_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_combined_bonafide_prosa_dir}")

print("-----------------------------------------------")
print(f"Total unseen UnID samples: {len(xlsr_combined_test_samples['unseen all samples (UnID)'])}")
print("-----------------------------------------------")

xlsr_combined_test_samples_dataloaders = {
    key: DataLoader(value, batch_size=batch_size, shuffle=False, num_workers=4)
    for key, value in xlsr_combined_test_samples.items()
}

## Test XLSR-Conformer TCM Model (Diff Pipeline)

In [ ]:
# Model configuration (same as training)
xlsr_diff_pipeline_model_config = {
    'emb_size': 144,           # Embedding size
    'num_encoders': 4,         # Number of conformer encoder blocks
    'heads': 4,                # Number of attention heads
    'kernel_size': 31,         # Kernel size for conv module
    'cp_path': 'xlsr2_300m.pt', # Path to pretrained XLSR model
    'fine_tune_ssl': True,     # Whether to fine-tune SSL model
    'nb_patho_features': 24    # Number of pathological features
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Set up model, optimizer, and scaler ---
model_diff = XLSRConformerTCMDiffPipeline(xlsr_diff_pipeline_model_config, device).to(device)
optimizer_diff = Adam(model_diff.parameters(), lr=learning_rate)
scaler_diff = GradScaler()

# --- Load the trained model ---
checkpoint_path_diff = r"pretrained_weights/diff_pipeline/XLSR_Conformer_TCM/xlsr_conformer_tcm_diff_pipeline-ep_30-bs_32-lr_1e-06.pth"

print(f"\nLoading model from: {checkpoint_path_diff}")
start_epoch_diff = load_model_xlsr_conformer_tcm(
    model_diff, optimizer_diff, scaler_diff,
    path=checkpoint_path_diff,
    device=device
)

print(f"Loaded checkpoint from epoch {start_epoch_diff}")

# Count parameters
nb_params_diff = sum([param.view(-1).size()[0] for param in model_diff.parameters() if param.requires_grad])
print(f'Number of trainable parameters: {nb_params_diff:,}')

In [ ]:
# Test the model
for key, test_sample in xlsr_combined_test_samples_dataloaders.items():
    print(f"\n{'='*60}")
    print(f"Testing {key}")
    print(f"{'='*60}")
    
    if key in main_keys:
        predictions, targets, metrics = test_xlsr_conformer_tcm(model_diff, test_sample, device=device)
        
        # Print summary
        print(f"\n--- Results Summary ---")
        print(f"Accuracy: {metrics['accuracy']:.2f}%")
        print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"F2 Score: {metrics['f2']:.4f}")
        print(f"EER: {metrics['eer']:.4f}")
        print(f"actDCF: {metrics['actDCF']:.4f}")
        print(f"minDCF: {metrics['minDCF']:.4f}")
        print(f"CLLR: {metrics['cllr']:.4f}")
    
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Testing completed!")
print(f"{'='*60}")